# Complementos de Investigação Operacional — capítulo 6: Filas de espera
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/applied-operations-research-course/blob/main/code/python/notebooks/06_queueing.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 6 of the course "Complements of Operational Research" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/applied-operations-research-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("applied-operations-research-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/applied-operations-research-course
    RAIZ = os.path.abspath(os.path.join("applied-operations-research-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 6.1 Filas de espera: conceitos, lei de Little e M/M/1

Exemplo [`ex06_1_mm1.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/06_queueing/06_1_mm1/ex06_1_mm1.py) · funções da UC: [`filas.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/06_queueing/06_1_mm1/filas.py)

```text
Reproduz os exemplos do deck 6.1 (conceitos, lei de Little e M/M/1).

Exemplo-guia: posto de carregamento rápido com um carregador, lambda = 3 carros/h, mu = 4/h:
    rho = 0.75, P0 = 0.25, L = 3, Lq = 2.25, W = 1 h, Wq = 45 min; Pn = 0.25*0.75^n;
    P(n>=1) = 75%, P(n>=3) = 42%, P(n>=8) = 10%; Little na loja (40/120 h = 20 min).
Efeito da utilização: lambda = 2, 2.5, 3, 3.5, 3.8 -> Wq = 15, 25, 45, 105, 285 min.
Cauda: 95% esperam menos de 2.71 h; a simulação (simula_mms(3, 4, 1), o código do slide)
dá 45.5 min, 0.75 e 2.72 h e compara-se com as fórmulas com tolerância.
Para resolver na aula: Exemplo 1 (controlo de bagagens, lambda = 10/min, mu = 12/min):
83.3%, 69.4%, 4.17 passageiros, 30 s.
```

In [2]:
import math

import numpy as np

from filas import mm1, simula_mms

ok = True
print("Complementos de IO — deck 6.1: filas de espera, lei de Little e M/M/1")

# ------------------------------------------------------------ posto de carregamento
lam, mu = 3, 4
m = mm1(lam, mu)
print("\nPosto de carregamento rápido (M/M/1, lambda = %g/h, mu = %g/h)" % (lam, mu))
print("  rho = %.4g  P0 = %.4g  L = %.4g  Lq = %.4g  W = %.4g h  Wq = %.4g min"
      % (m['rho'], m['P0'], m['L'], m['Lq'], m['W'], m['Wq'] * 60))
print("  Little: lambda*W = %.4g (= L);  lambda*Wq = %.4g (= Lq)" % (lam * m['W'], lam * m['Wq']))
Pn = np.array([m['Pn'](n) for n in range(9)])
print("  Pn, n = 0..8: %s" % " ".join("%.4f" % x for x in Pn))
pmaior = [m['rho'] ** k for k in (1, 3, 8)]          # P(n >= k) = rho^k
print("  P(n>=1) = %.0f%%  P(n>=3) = %.0f%%  P(n>=8) = %.0f%%" % tuple(100 * p for p in pmaior))
print("Little na loja: 40 clientes, 120 entradas/h -> %.4g min por cliente" % (40 / 120 * 60))
ok &= (np.allclose([m['rho'], m['P0'], m['L'], m['Lq'], m['W'], m['Wq'] * 60], [0.75, 0.25, 3, 2.25, 1, 45])
       and np.allclose(np.round(Pn, 4), [0.25, 0.1875, 0.1406, 0.1055, 0.0791, 0.0593, 0.0445, 0.0334, 0.025])
       and np.allclose(np.round(pmaior, 2), [0.75, 0.42, 0.10]))

# ------------------------------------------------------------ efeito da utilização
print("\nO efeito da utilização (mu = 4/h)")
Wq = []
for l in (2, 2.5, 3, 3.5, 3.8):
    x = mm1(l, mu)
    Wq.append(x['Wq'] * 60)
    print("  lambda = %.1f/h  rho = %.3f  Wq = %.4g min" % (l, x['rho'], Wq[-1]))
print("  3 -> 3.5 carros/h (+%.0f%%): Wq passa de %.4g para %.4g min (x%.2f)"
      % (100 * 0.5 / 3, Wq[2], Wq[3], Wq[3] / Wq[2]))
ok &= np.allclose(Wq, [15, 25, 45, 105, 285])

# ------------------------------------------------------------ cauda e simulação
t95 = math.log(m['rho'] / 0.05) / (mu - lam)          # rho*exp(-(mu - lam) t) = 0.05
print("\nCauda: P(Wq > t) = rho*exp(-(mu - lambda)*t); 95%% esperam menos de %.2f h" % t95)
sim = simula_mms(lam, mu, 1)
print("Simulação (simula_mms(3, 4, 1): 200000 carros, semente 1, sem os 10000 primeiros)")
print("  Wq = %.1f min (fórmula %.4g)  P(esperar) = %.2f (%.2f)  quantil 95%% = %.2f h (%.2f)"
      % (sim['Wq'] * 60, m['Wq'] * 60, sim['Pw'], m['rho'], sim['p95'], t95))
tol = (abs(sim['Wq'] - m['Wq']) <= 0.10 * m['Wq'] and abs(sim['Pw'] - m['rho']) <= 0.02
       and abs(sim['p95'] - t95) <= 0.10 * t95)
print("  simulação dentro da tolerância (10%% nas médias e no quantil, 0.02 nas probabilidades): %s"
      % ("sim" if tol else "não"))
ok &= round(t95, 2) == 2.71 and tol

# ------------------------------------------------------------ Exemplo 1 (para resolver na aula)
b = mm1(10, 12)
print("\nExemplo 1 — controlo de bagagens (lambda = 10/min, mu = 12/min)")
print("  a) P(máquina não ociosa) = rho = %.4f (%.1f%%)" % (b['rho'], 100 * b['rho']))
print("  b) P(haver fila) = P(n>=2) = rho^2 = %.4f (%.1f%%)" % (b['rho'] ** 2, 100 * b['rho'] ** 2))
print("  c) Lq = %.4f passageiros" % b['Lq'])
print("  d) W = %.4g min = %.4g s (Wq = %.4g s mais o controlo)" % (b['W'], b['W'] * 60, b['Wq'] * 60))
ok &= (round(100 * b['rho'], 1) == 83.3 and round(100 * b['rho'] ** 2, 1) == 69.4
       and abs(b['Lq'] - 25 / 6) < 1e-9 and abs(b['W'] * 60 - 30) < 1e-9)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 6.1: filas de espera, lei de Little e M/M/1

Posto de carregamento rápido (M/M/1, lambda = 3/h, mu = 4/h)
  rho = 0.75  P0 = 0.25  L = 3  Lq = 2.25  W = 1 h  Wq = 45 min
  Little: lambda*W = 3 (= L);  lambda*Wq = 2.25 (= Lq)
  Pn, n = 0..8: 0.2500 0.1875 0.1406 0.1055 0.0791 0.0593 0.0445 0.0334 0.0250
  P(n>=1) = 75%  P(n>=3) = 42%  P(n>=8) = 10%
Little na loja: 40 clientes, 120 entradas/h -> 20 min por cliente

O efeito da utilização (mu = 4/h)
  lambda = 2.0/h  rho = 0.500  Wq = 15 min
  lambda = 2.5/h  rho = 0.625  Wq = 25 min
  lambda = 3.0/h  rho = 0.750  Wq = 45 min
  lambda = 3.5/h  rho = 0.875  Wq = 105 min
  lambda = 3.8/h  rho = 0.950  Wq = 285 min
  3 -> 3.5 carros/h (+17%): Wq passa de 45 para 105 min (x2.33)

Cauda: P(Wq > t) = rho*exp(-(mu - lambda)*t); 95% esperam menos de 2.71 h


Simulação (simula_mms(3, 4, 1): 200000 carros, semente 1, sem os 10000 primeiros)
  Wq = 45.5 min (fórmula 45)  P(esperar) = 0.75 (0.75)  quantil 95% = 2.72 h (2.71)
  simulação dentro da tolerância (10% nas médias e no quantil, 0.02 nas probabilidades): sim

Exemplo 1 — controlo de bagagens (lambda = 10/min, mu = 12/min)
  a) P(máquina não ociosa) = rho = 0.8333 (83.3%)
  b) P(haver fila) = P(n>=2) = rho^2 = 0.6944 (69.4%)
  c) Lq = 4.1667 passageiros
  d) W = 0.5 min = 30 s (Wq = 25 s mais o controlo)

confere com os slides: sim


## 6.2 Filas de espera: capacidade limitada e vários servidores

Exemplo [`ex06_2_mmk_mms.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/06_queueing/06_2_mmk_mms/ex06_2_mmk_mms.py) · funções da UC: [`filas.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/06_queueing/06_1_mm1/filas.py), [`filas_mms.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/06_queueing/06_2_mmk_mms/filas_mms.py)

```text
Reproduz os exemplos do deck 6.2 (capacidade limitada e vários servidores).

M/M/1/K no posto de 6.1 (lambda = 3/h, mu = 4/h), K = 2..6 e infinito: P_K, lambda_ef, L, W;
com K = 4, 10.4% desistem (0.31 por hora) e quem fica está 32 min; simula_mm1k(3, 4, 4) compara-se
com as fórmulas com tolerância.
M/M/s com lambda = 6/h: s = 2..5 -> P(esperar) 0.643, 0.237, 0.075, 0.020 e Wq 19.3, 2.4, 0.45, 0.09 min.
Fila única (M/M/2, 19.3 min) contra duas filas separadas (duas M/M/1, 45 min).
Custo 12 s + 20 Lq (custo_caixas): 62.57, 40.74, 48.90, 60.17 €/h -> 3 carregadores;
nível de serviço «menos de 10% esperam» -> 4 (7.5%), mais 8.16 €/h. simula_mms(6, 4, 3): Wq ~ 2.4 min.
Para resolver na aula: Exemplo 2 (restaurante, M/M/1/K com K = 300): Lq = 4/3, P(fila) = 44.4%,
W = 18 s, igual ao M/M/1 (P_K ~ 5e-54).
```

In [3]:
import numpy as np

from filas import mm1, simula_mms
from filas_mms import mm1k, mms, custo_caixas, simula_mm1k


def confere(v, alvo, casas):
    """v arredondado a `casas` casas decimais (por coluna) dá os valores do slide?"""
    v, alvo = np.asarray(v, float), np.asarray(alvo, float)
    return bool(np.all(np.abs(v - alvo) <= 0.5 * 10.0 ** -np.asarray(casas, float) + 1e-9))


ok = True
print("Complementos de IO — deck 6.2: capacidade limitada e vários servidores")

# ------------------------------------------------------------ M/M/1/K
lam, mu = 3, 4
print("\nM/M/1/K: posto de carregamento com K lugares (lambda = 3/h, mu = 4/h)")
print("  K    P_K    lambda_ef    L     W (min)")
tab = []
for K in (2, 3, 4, 5, 6):
    x = mm1k(lam, mu, K)
    tab.append([x['PK'], x['lef'], x['L'], x['W'] * 60])
    print("  %-3d  %.3f  %.2f         %.2f  %.0f" % (K, x['PK'], x['lef'], x['L'], x['W'] * 60))
m = mm1(lam, mu)
print("  inf  0      %.4g            %.4g     %.4g" % (lam, m['L'], m['W'] * 60))
k4 = mm1k(lam, mu, 4)
print("  K = 4: desistem %.1f%% (%.2f por hora); quem fica está %.0f min"
      % (100 * k4['PK'], lam * k4['PK'], k4['W'] * 60))
ok &= confere(tab, [[0.243, 2.27, 0.81, 21], [0.154, 2.54, 1.15, 27], [0.104, 2.69, 1.44, 32],
                   [0.072, 2.78, 1.70, 37], [0.051, 2.85, 1.92, 41]], [3, 2, 2, 0])
ok &= round(lam * k4['PK'], 2) == 0.31

sk = simula_mm1k(lam, mu, 4)
print("Simulação (simula_mm1k(3, 4, 4): 200000 chegadas, semente 1)")
print("  desistem %.1f%% (fórmula %.1f%%)  W = %.0f min (%.0f)"
      % (100 * sk['PK'], 100 * k4['PK'], sk['W'] * 60, k4['W'] * 60))
tol = abs(sk['PK'] - k4['PK']) <= 0.02 and abs(sk['W'] - k4['W']) <= 0.10 * k4['W']
print("  simulação dentro da tolerância (10%% nas médias, 0.02 nas probabilidades): %s" % ("sim" if tol else "não"))
ok &= tol

# ------------------------------------------------------------ M/M/s
lam = 6
print("\nM/M/s: a procura duplica (lambda = 6/h, mu = 4/h)")
print("  s  P(esperar)  Lq     Wq (min)")
v = []
for s in (2, 3, 4, 5):
    y = mms(lam, mu, s)
    v.append([y['Pw'], y['Lq'], y['Wq'] * 60])
    print("  %d  %.3f       %.3f  %.2f" % (s, y['Pw'], y['Lq'], y['Wq'] * 60))
v = np.array(v)
ok &= (confere(v[:, 0], [0.643, 0.237, 0.075, 0.020], 3)
       and confere(v[:, 1], [1.93, 0.24, 0.045, 0.009], [2, 2, 3, 3])
       and confere(v[:, 2], [19.3, 2.4, 0.45, 0.09], [1, 1, 2, 2]))

# ------------------------------------------------------------ fila única ou separadas
sep, uni = mm1(3, mu)['Wq'] * 60, mms(6, mu, 2)['Wq'] * 60
print("\nFila única ou uma fila por carregador (2 carregadores, lambda = 6/h)")
print("  duas filas separadas (duas M/M/1, lambda = 3): Wq = %.1f min" % sep)
print("  uma fila única (M/M/2, lambda = 6):            Wq = %.1f min" % uni)
ok &= round(sep, 1) == 45 and round(uni, 1) == 19.3

# ------------------------------------------------------------ custo contra espera
print("\nQuantos carregadores? custo(s) = 12 s + 20 Lq(s) (€/h)")
print("  s  carregadores  espera  total")
tc = custo_caixas(lam, mu, 12, 20, 5)
for s, Pw, Lq, L, c in tc:
    print("  %d  %-12.0f  %-6.2f  %.2f" % (s, 12 * s, 20 * Lq, c))
c = np.array([r[4] for r in tc])
smin = tc[int(c.argmin())][0]
iserv = next(i for i, r in enumerate(tc) if r[1] < 0.10)
sserv = tc[iserv][0]
print("  mínimo custo: %d carregadores (%.2f €/h); esperam %.1f%%"
      % (smin, c.min(), 100 * tc[int(c.argmin())][1]))
print("  menos de 10%% esperam: %d carregadores (%.1f%%), %.2f €/h; preço do nível de serviço %.2f €/h"
      % (sserv, 100 * tc[iserv][1], c[iserv], c[iserv] - c.min()))
ok &= ([r[0] for r in tc] == [2, 3, 4, 5] and confere(c, [62.57, 40.74, 48.90, 60.17], 2)
       and smin == 3 and sserv == 4 and round(c[iserv] - c.min(), 2) == 8.16)

sm = simula_mms(lam, mu, 3)
y3 = mms(lam, mu, 3)
print("Simulação (simula_mms(6, 4, 3), semente 1)")
print("  Wq = %.2f min (fórmula %.2f)  P(esperar) = %.3f (%.3f)"
      % (sm['Wq'] * 60, y3['Wq'] * 60, sm['Pw'], y3['Pw']))
tol = abs(sm['Wq'] - y3['Wq']) <= 0.10 * y3['Wq'] and abs(sm['Pw'] - y3['Pw']) <= 0.02
print("  simulação dentro da tolerância (10%% nas médias, 0.02 nas probabilidades): %s" % ("sim" if tol else "não"))
ok &= tol

# ------------------------------------------------------------ Exemplo 2 (para resolver na aula)
z = mm1k(400, 600, 300)
zi = mm1(400, 600)
pfila = 1 - z['P'][0] - z['P'][1]
print("\nExemplo 2 — restaurante (M/M/1/GD/300/inf, lambda = 400/h, mu = 600/h)")
print("  a) Lq = %.4f clientes" % z['Lq'])
print("  b) P(haver fila) = 1 - P0 - P1 = %.4f (%.1f%%)" % (pfila, 100 * pfila))
print("  c) W = %.4g s" % (z['W'] * 3600))
print("  d) M/M/1 sem limite: Lq = %.4f, P(fila) = rho^2 = %.4f, W = %.4g s; P_K = %.1e: o limite não conta"
      % (zi['Lq'], zi['rho'] ** 2, zi['W'] * 3600, z['PK']))
ok &= (abs(z['Lq'] - 4 / 3) < 1e-9 and abs(pfila - 4 / 9) < 1e-9 and abs(z['W'] * 3600 - 18) < 1e-9
       and abs(zi['Lq'] - z['Lq']) < 1e-9 and z['PK'] < 1e-50)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 6.2: capacidade limitada e vários servidores

M/M/1/K: posto de carregamento com K lugares (lambda = 3/h, mu = 4/h)
  K    P_K    lambda_ef    L     W (min)
  2    0.243  2.27         0.81  21
  3    0.154  2.54         1.15  27
  4    0.104  2.69         1.44  32
  5    0.072  2.78         1.70  37
  6    0.051  2.85         1.92  41
  inf  0      3            3     60
  K = 4: desistem 10.4% (0.31 por hora); quem fica está 32 min


Simulação (simula_mm1k(3, 4, 4): 200000 chegadas, semente 1)
  desistem 10.1% (fórmula 10.4%)  W = 32 min (32)
  simulação dentro da tolerância (10% nas médias, 0.02 nas probabilidades): sim

M/M/s: a procura duplica (lambda = 6/h, mu = 4/h)
  s  P(esperar)  Lq     Wq (min)
  2  0.643       1.929  19.29
  3  0.237       0.237  2.37
  4  0.075       0.045  0.45
  5  0.020       0.009  0.09

Fila única ou uma fila por carregador (2 carregadores, lambda = 6/h)
  duas filas separadas (duas M/M/1, lambda = 3): Wq = 45.0 min
  uma fila única (M/M/2, lambda = 6):            Wq = 19.3 min

Quantos carregadores? custo(s) = 12 s + 20 Lq(s) (€/h)
  s  carregadores  espera  total
  2  24            38.57   62.57
  3  36            4.74    40.74
  4  48            0.90    48.90
  5  60            0.17    60.17
  mínimo custo: 3 carregadores (40.74 €/h); esperam 23.7%
  menos de 10% esperam: 4 carregadores (7.5%), 48.90 €/h; preço do nível de serviço 8.16 €/h
Simulação (simula_mms(6, 4, 3), semente 

## 6.3 Filas de espera: população finita (M/M/R)

Exemplo [`ex06_3_mmr.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/06_queueing/06_3_mmr/ex06_3_mmr.py) · funções da UC: [`filas_mmr.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/06_queueing/06_3_mmr/filas_mmr.py)

```text
Reproduz os exemplos do deck 6.3 (população finita, M/M/R).

Exemplo-guia: K = 10 empilhadores, lambda = 0.1 avarias/dia, mu = 0.8 reparações/dia;
técnico 150 €/dia, empilhador parado 250 €/dia. R = 1, 2, 3:
    P0 0.122, 0.283, 0.305; L 2.97, 1.35, 1.14; Lq 2.09, 0.26, 0.04; W 4.23, 1.55, 1.29 dias;
    custo 893, 636, 736 €/dia -> dois técnicos (menos 257 €/dia); com um técnico Wq ~ 2.98 dias;
    o 3.º técnico tira 0.2 empilhadores da paragem (50 €/dia) e custa 150.
População infinita: lambda = 1/dia contra mu = 0.8 dá rho = 1.25 (instável); na lavandaria 1 contra 0.4.
Para resolver na aula: Exemplo 3 (lavandaria, 5 máquinas, 3 mecânicos contra um super-mecânico):
L = 1.71 contra 1.16 -> trocar; Wq 0.11 contra 0.68 dias, W 2.61 contra 1.51 dias.
```

In [4]:
import numpy as np

from filas_mmr import mmR


def confere(v, alvo, casas):
    """v arredondado a `casas` casas decimais (por coluna) dá os valores do slide?"""
    v, alvo = np.asarray(v, float), np.asarray(alvo, float)
    return bool(np.all(np.abs(v - alvo) <= 0.5 * 10.0 ** -np.asarray(casas, float) + 1e-9))


ok = True
print("Complementos de IO — deck 6.3: população finita (M/M/R)")

# ------------------------------------------------------------ empilhadores
K, lam, mu, ct, cp = 10, 0.1, 0.8, 150, 250
print("\nEmpilhadores (K = %d, lambda = %g/dia, mu = %g/dia; técnico %d €/dia, parado %d €/dia)"
      % (K, lam, mu, ct, cp))
print("  R  P0     L (parados)  Lq    a funcionar  W (dias)  custo (€/dia)")
res, tab = [], []
for R in (1, 2, 3):
    e = mmR(K, R, lam, mu)
    custo = ct * R + cp * e['L']
    res.append(e)
    tab.append([e['P0'], e['L'], e['Lq'], e['a_funcionar'], e['W'], custo])
    print("  %d  %.3f  %.2f         %.2f  %.2f         %.2f      %.0f"
          % (R, e['P0'], e['L'], e['Lq'], e['a_funcionar'], e['W'], custo))
tab = np.array(tab)
best = int(tab[:, 5].argmin()) + 1
print("  decisão: %d técnicos (%.0f €/dia), menos %.0f €/dia do que com um só"
      % (best, tab[best - 1, 5], tab[0, 5] - tab[best - 1, 5]))
print("  com um técnico, Wq = W - 1/mu = %.2f dias" % res[0]['Wq'])
dL = res[1]['L'] - res[2]['L']
print("  o 3.º técnico tira %.1f empilhadores da paragem (%.1f €/dia) e custa %d €/dia" % (dL, cp * dL, ct))
ok &= (confere(tab, [[0.122, 2.97, 2.09, 7.03, 4.23, 893], [0.283, 1.35, 0.26, 8.65, 1.55, 636],
                    [0.305, 1.14, 0.04, 8.86, 1.29, 736]], [3, 2, 2, 2, 2, 0])
       and best == 2 and round(tab[0, 5] - tab[1, 5]) == 257 and round(res[0]['Wq'], 2) == 2.98
       and round(dL, 1) == 0.2 and round(cp * dL, -1) == 50)

# ------------------------------------------------------------ população infinita?
print("\nCom o modelo de população infinita: lambda = %g x %g = %g avarias/dia, mu = %g -> rho = %.2f (instável)"
      % (K, lam, K * lam, mu, K * lam / mu))
print("Na lavandaria: lambda = 5 x 0.2 = %g por dia contra mu = 0.4 de cada mecânico" % (5 * 0.2))
ok &= abs(K * lam / mu - 1.25) < 1e-12

# ------------------------------------------------------------ Exemplo 3 (para resolver na aula)
l3 = mmR(5, 3, 0.2, 1 / 2.5)
l1 = mmR(5, 1, 0.2, 1 / (5 / 6))
print("\nExemplo 3 — lavandaria (5 máquinas, lambda = 0.2/dia)")
print("  opção                        P0      L       Lq      a funcionar  W (dias)  Wq (dias)")
for nome, l in (("3 mecânicos (mu = 0.4)     ", l3), ("super-mecânico (mu = 1.2)  ", l1)):
    print("  %s  %.4f  %.4f  %.4f  %.4f       %.4f    %.4f"
          % (nome, l['P0'], l['L'], l['Lq'], l['a_funcionar'], l['W'], l['Wq']))
print("  decisão (mesmo salário): %s — menos máquinas paradas (%.2f contra %.2f)"
      % ("trocar pelo super-mecânico" if l1['L'] < l3['L'] else "manter os 3 mecânicos", l1['L'], l3['L']))
ok &= (round(l3['L'], 2) == 1.71 and round(l1['L'], 2) == 1.16 and l1['L'] < l3['L']
       and round(l3['Wq'], 2) == 0.11 and round(l1['Wq'], 2) == 0.68
       and round(l3['W'], 2) == 2.61 and round(l1['W'], 2) == 1.51)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 6.3: população finita (M/M/R)

Empilhadores (K = 10, lambda = 0.1/dia, mu = 0.8/dia; técnico 150 €/dia, parado 250 €/dia)
  R  P0     L (parados)  Lq    a funcionar  W (dias)  custo (€/dia)
  1  0.122  2.97         2.09  7.03         4.23      893
  2  0.283  1.35         0.26  8.65         1.55      636
  3  0.305  1.14         0.04  8.86         1.29      736
  decisão: 2 técnicos (636 €/dia), menos 257 €/dia do que com um só
  com um técnico, Wq = W - 1/mu = 2.98 dias
  o 3.º técnico tira 0.2 empilhadores da paragem (50.7 €/dia) e custa 150 €/dia

Com o modelo de população infinita: lambda = 10 x 0.1 = 1 avarias/dia, mu = 0.8 -> rho = 1.25 (instável)
Na lavandaria: lambda = 5 x 0.2 = 1 por dia contra mu = 0.4 de cada mecânico

Exemplo 3 — lavandaria (5 máquinas, lambda = 0.2/dia)
  opção                        P0      L       Lq      a funcionar  W (dias)  Wq (dias)
  3 mecânicos (mu = 0.4)       0.1293  1.7145  0.0718  3.2855       2.6093    0.1093
  super